In [54]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path 
import sys,importlib
import re,ast
import joblib
import json , shutil
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

In [5]:
ROOT = Path("/workspace")
excel = ROOT / "data/raw/02_CortexHR_AI_ML_Dataset.xlsx"

In [6]:
xls = pd.ExcelFile(excel)

In [10]:
salary_df = pd.read_excel(excel,sheet_name="salary_training")

In [11]:
 salary_df.shape

(2934, 17)

In [12]:
salary_df.columns.tolist()

['hire_id',
 'hire_date',
 'hire_year',
 'department_id',
 'role_id',
 'job_level',
 'band_min_2026',
 'band_mid_2026',
 'band_max_2026',
 'prior_experience_years',
 'career_gap_months',
 'work_location',
 'highest_education',
 'skills_count',
 'market_index',
 'offered_annual_fixed',
 'source']

In [13]:
salary_df

,hire_id,hire_date,hire_year,department_id,role_id,job_level,band_min_2026,band_mid_2026,band_max_2026,prior_experience_years,career_gap_months,work_location,highest_education,skills_count,market_index,offered_annual_fixed,source
0,EMP-0001,2012-01-19,2012,D01,R002,1,400000,500000,650000,0.0,0,Pune,B.E. Computer Science,5,0.4676,240480,master_hire
1,EMP-0002,2012-01-31,2012,D01,R007,4,2000000,2500000,3250000,17.1,9,Pune,B.E. Computer Science,4,0.4684,1312440,master_hire
2,EMP-0003,2012-03-08,2012,D09,R051,2,360000,450000,585000,3.0,0,Coimbatore,B.Sc,6,0.4709,202836,master_hire
3,EMP-0004,2012-03-20,2012,D03,R017,3,1200000,1500000,1950000,4.0,24,Erode,MCA,5,0.4718,634128,master_hire
4,EMP-0005,2012-03-28,2012,D06,R033,1,208000,260000,338000,0.0,9,Bengaluru,B.E. Electronics,4,0.4723,114432,master_hire
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2929,HIRE-2196,2023-04-21,2023,D04,R025,4,1760000,2200000,2860000,7.5,0,Hyderabad,B.Com,8,0.8541,1559028,historical_extension
2930,HIRE-2197,2019-09-29,2019,D01,R004,2,600000,750000,975000,1.3,0,Bengaluru,B.Tech Information Technology,5,0.7059,555780,historical_extension
2931,HIRE-2198,2026-07-09,2026,D01,R005,3,1280000,1600000,2080000,7.5,0,Bengaluru,MCA,7,1.0146,1828116,historical_extension
2932,HIRE-2199,2024-03-29,2024,D06,R033,1,208000,260000,338000,0.5,0,Pune,B.Sc,4,0.8981,224664,historical_extension


In [16]:
protected_df = pd.read_excel(excel,sheet_name="protected_attributes")

In [17]:
protected_df.shape

(5317, 5)

In [18]:
protected_df.columns.tolist()

['person_ref', 'record_type', 'gender', 'age_years', 'age_band']

In [19]:
protected_df.head()

,person_ref,record_type,gender,age_years,age_band
0,EMP-0002,attrition_snapshot,Male,52.3,45+
1,EMP-0004,attrition_snapshot,Male,41.3,35-44
2,EMP-0011,attrition_snapshot,Male,34.4,25-34
3,EMP-0012,attrition_snapshot,Female,40.2,35-44
4,EMP-0013,attrition_snapshot,Male,39.3,35-44


========== SALARY TRAINING ==========
Shape: (2934, 17)
Columns: ['hire_id', 'hire_date', 'hire_year', 'department_id', 'role_id', 'job_level', 'band_min_2026', 'band_mid_2026', 'band_max_2026', 'prior_experience_years', 'career_gap_months', 'work_location', 'highest_education', 'skills_count', 'market_index', 'offered_annual_fixed', 'source']


,hire_id,hire_date,hire_year,department_id,role_id,job_level,band_min_2026,band_mid_2026,band_max_2026,prior_experience_years,career_gap_months,work_location,highest_education,skills_count,market_index,offered_annual_fixed,source
0,EMP-0001,2012-01-19,2012,D01,R002,1,400000,500000,650000,0.0,0,Pune,B.E. Computer Science,5,0.4676,240480,master_hire
1,EMP-0002,2012-01-31,2012,D01,R007,4,2000000,2500000,3250000,17.1,9,Pune,B.E. Computer Science,4,0.4684,1312440,master_hire
2,EMP-0003,2012-03-08,2012,D09,R051,2,360000,450000,585000,3.0,0,Coimbatore,B.Sc,6,0.4709,202836,master_hire
3,EMP-0004,2012-03-20,2012,D03,R017,3,1200000,1500000,1950000,4.0,24,Erode,MCA,5,0.4718,634128,master_hire
4,EMP-0005,2012-03-28,2012,D06,R033,1,208000,260000,338000,0.0,9,Bengaluru,B.E. Electronics,4,0.4723,114432,master_hire



========== PROTECTED ATTRIBUTES ==========
Shape: (5317, 5)
Columns: ['person_ref', 'record_type', 'gender', 'age_years', 'age_band']


,person_ref,record_type,gender,age_years,age_band
0,EMP-0002,attrition_snapshot,Male,52.3,45+
1,EMP-0004,attrition_snapshot,Male,41.3,35-44
2,EMP-0011,attrition_snapshot,Male,34.4,25-34
3,EMP-0012,attrition_snapshot,Female,40.2,35-44
4,EMP-0013,attrition_snapshot,Male,39.3,35-44


In [27]:
display(protected_df["record_type"].value_counts(dropna=False).rename("count").to_frame())
print("\nPROTECTED ATTRIBUTE DUPLICATES")
print("Duplicate person_ref:", protected_df["person_ref"].duplicated().sum())
print(
    "Duplicate person_ref + record_type:",
    protected_df.duplicated(["person_ref", "record_type"]).sum()
)

salary_ids = set(salary_df["hire_id"].astype(str))
protected_ids = set(protected_df["person_ref"].astype(str))
matched_ids = salary_ids.intersection(protected_ids)

PROTECTED ATTRIBUTE RECORD TYPES


,count
record_type,
salary_hire,2934
attrition_snapshot,2383



PROTECTED ATTRIBUTE DUPLICATES
Duplicate person_ref: 383
Duplicate person_ref + record_type: 0


In [23]:
len(salary_ids)

2934

In [24]:
len(salary_df)

2934

In [25]:
len(protected_ids)

4934

In [26]:
len(matched_ids)

2934

In [22]:
display(salary_df["career_gap_months"].describe().to_frame())

,career_gap_months
count,2934.000000
mean,1.829243
std,5.997484
min,0.000000
25%,0.000000
50%,0.000000
75%,0.000000
max,36.000000


In [ ]:
#join the gender information

In [28]:
salary_protected = protected_df[protected_df["record_type"] == "salary_hire"].copy()
salary_protected = salary_protected[["person_ref", "gender", "age_years", "age_band"]]
salary_audit_df = salary_df.merge(
    salary_protected,
    left_on="hire_id",
    right_on="person_ref",
    how="left",
    validate="one_to_one"
)
print("\nMissing protected attributes:")
print("\nCareer gap groups:")
salary_audit_df["career_gap_group"] = np.where(
    salary_audit_df["career_gap_months"] == 0,
    "No career gap",
    "Career gap"
)
display(salary_audit_df.head())

Merged dataset shape: (2934, 21)

Gender distribution:


,count
gender,
Male,1652
Female,1222
Other,60



Missing protected attributes:


gender       0
age_years    0
age_band     0
dtype: int64


Career gap groups:


,count
career_gap_group,
No career gap,2549
Career gap,385


,hire_id,hire_date,hire_year,department_id,role_id,job_level,band_min_2026,band_mid_2026,band_max_2026,prior_experience_years,...,highest_education,skills_count,market_index,offered_annual_fixed,source,person_ref,gender,age_years,age_band,career_gap_group
0,EMP-0001,2012-01-19,2012,D01,R002,1,400000,500000,650000,0.0,...,B.E. Computer Science,5,0.4676,240480,master_hire,EMP-0001,Male,22.3,<25,No career gap
1,EMP-0002,2012-01-31,2012,D01,R007,4,2000000,2500000,3250000,17.1,...,B.E. Computer Science,4,0.4684,1312440,master_hire,EMP-0002,Male,38.6,35-44,Career gap
2,EMP-0003,2012-03-08,2012,D09,R051,2,360000,450000,585000,3.0,...,B.Sc,6,0.4709,202836,master_hire,EMP-0003,Male,26.8,25-34,No career gap
3,EMP-0004,2012-03-20,2012,D03,R017,3,1200000,1500000,1950000,4.0,...,MCA,5,0.4718,634128,master_hire,EMP-0004,Male,27.8,25-34,Career gap
4,EMP-0005,2012-03-28,2012,D06,R033,1,208000,260000,338000,0.0,...,B.E. Electronics,4,0.4723,114432,master_hire,EMP-0005,Female,24.8,<25,Career gap


In [29]:
 salary_audit_df.shape

(2934, 22)

In [32]:
display(salary_audit_df["gender"].value_counts(dropna=False).to_frame("count"))


,count
gender,
Male,1652
Female,1222
Other,60


In [33]:
display(salary_audit_df[["gender", "age_years", "age_band"]].isna().sum())


gender       0
age_years    0
age_band     0
dtype: int64

In [35]:
salary_audit_df["career_gap_group"] = np.where(
    salary_audit_df["career_gap_months"] == 0,
    "No career gap",
    "Career gap")

In [36]:

display(
    salary_audit_df["career_gap_group"]
    .value_counts()
    .to_frame("count")
)

,count
career_gap_group,
No career gap,2549
Career gap,385


In [37]:
salary_audit_df.head()

,hire_id,hire_date,hire_year,department_id,role_id,job_level,band_min_2026,band_mid_2026,band_max_2026,prior_experience_years,...,highest_education,skills_count,market_index,offered_annual_fixed,source,person_ref,gender,age_years,age_band,career_gap_group
0,EMP-0001,2012-01-19,2012,D01,R002,1,400000,500000,650000,0.0,...,B.E. Computer Science,5,0.4676,240480,master_hire,EMP-0001,Male,22.3,<25,No career gap
1,EMP-0002,2012-01-31,2012,D01,R007,4,2000000,2500000,3250000,17.1,...,B.E. Computer Science,4,0.4684,1312440,master_hire,EMP-0002,Male,38.6,35-44,Career gap
2,EMP-0003,2012-03-08,2012,D09,R051,2,360000,450000,585000,3.0,...,B.Sc,6,0.4709,202836,master_hire,EMP-0003,Male,26.8,25-34,No career gap
3,EMP-0004,2012-03-20,2012,D03,R017,3,1200000,1500000,1950000,4.0,...,MCA,5,0.4718,634128,master_hire,EMP-0004,Male,27.8,25-34,Career gap
4,EMP-0005,2012-03-28,2012,D06,R033,1,208000,260000,338000,0.0,...,B.E. Electronics,4,0.4723,114432,master_hire,EMP-0005,Female,24.8,<25,Career gap


In [46]:
display(
    salary_audit_df["offered_annual_fixed"]
    .describe()
    .to_frame("value")
)

,value
count,2.934000e+03
mean,8.344280e+05
std,6.413154e+05
min,1.144320e+05
25%,3.805410e+05
50%,6.241920e+05
75%,1.114299e+06
max,4.197972e+06


In [39]:
candidate_features = [
    "department_id",
    "role_id",
    "job_level",
    "prior_experience_years",
    "work_location",
    "highest_education",
    "skills_count",
    "market_index"
]

In [40]:
display(salary_audit_df[candidate_features + ["offered_annual_fixed"]].isna().sum().to_frame("missing_count"))

,missing_count
department_id,0
role_id,0
job_level,0
prior_experience_years,0
work_location,0
highest_education,0
skills_count,0
market_index,0
offered_annual_fixed,0


In [41]:
display(
    salary_audit_df[
        [
            "band_min_2026",
            "band_mid_2026",
            "band_max_2026",
            "offered_annual_fixed"
        ]
    ].describe().round(2)
)

,band_min_2026,band_mid_2026,band_max_2026,offered_annual_fixed
count,2934.00,2934.0,2934.00,2934.00
mean,820654.40,1025818.0,1333563.39,834428.05
std,605416.08,756770.1,983801.13,641315.38
min,208000.00,260000.0,338000.00,114432.00
25%,400000.00,500000.0,650000.00,380541.00
50%,600000.00,750000.0,975000.00,624192.00
75%,1200000.00,1500000.0,1950000.00,1114299.00
max,3040000.00,3800000.0,4940000.00,4197972.00


In [43]:
salary_audit_df["within_existing_band"] = (salary_audit_df["offered_annual_fixed"].between(salary_audit_df["band_min_2026"],salary_audit_df["band_max_2026"]))

In [44]:
display(salary_audit_df["within_existing_band"].value_counts().to_frame("count"))

,count
within_existing_band,
True,1520
False,1414


In [45]:
display(salary_audit_df["career_gap_months"].describe().to_frame("months"))

,months
count,2934.000000
mean,1.829243
std,5.997484
min,0.000000
25%,0.000000
50%,0.000000
75%,0.000000
max,36.000000


In [47]:
target = "offered_annual_fixed"
categorical_features = [
    "department_id",
    "role_id",
    "job_level",
    "work_location",
    "highest_education"
]
numerical_features = [
    "prior_experience_years",
    "skills_count",
    "market_index"
]
selected_features = categorical_features + numerical_features
X = salary_audit_df[selected_features].copy()
y = salary_audit_df[target].copy()
print("Selected features:", selected_features)
print("X shape:", X.shape)
print("Target shape:", y.shape)
print("\nCategorical features:")
print(categorical_features)
print("\nNumerical features:")
print(numerical_features)
print("\nProtected attributes excluded:", 
      ["gender", "age_years", "age_band"])
print("Career gap excluded from model:", "career_gap_months")
print("Existing salary bands excluded from model.")

Selected features: ['department_id', 'role_id', 'job_level', 'work_location', 'highest_education', 'prior_experience_years', 'skills_count', 'market_index']
X shape: (2934, 8)
Target shape: (2934,)

Categorical features:
['department_id', 'role_id', 'job_level', 'work_location', 'highest_education']

Numerical features:
['prior_experience_years', 'skills_count', 'market_index']

Protected attributes excluded: ['gender', 'age_years', 'age_band']
Career gap excluded from model: career_gap_months
Existing salary bands excluded from model.


In [49]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42)
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

X_train: (2347, 8)
y_train: (2347,)


In [50]:
len(X_train) + len(X_test)

2934

In [52]:
round(len(X_train) / len(X) * 100, 2)
round(len(X_test) / len(X) * 100, 2)

20.01

In [53]:
 X_test.index.equals(y_test.index)

True

In [56]:
categorical_pipeline = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),("encoder", OneHotEncoder(handle_unknown="ignore"))])

In [57]:
numerical_pipeline = Pipeline([    ("imputer", SimpleImputer(strategy="median")),("scaler", StandardScaler())])

In [59]:
preprocessor = ColumnTransformer(
    transformers=[("categorical", categorical_pipeline, categorical_features),("numerical", numerical_pipeline, numerical_features)],sparse_threshold=0)

In [61]:
task15_models = {
    "Linear Regression": Pipeline([
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]),
    "Gradient Boosting": Pipeline([
        ("preprocessor", preprocessor),
        ("model", GradientBoostingRegressor(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=2,
            random_state=42
        ))
    ])
}

In [62]:
role_mean_salary = (pd.DataFrame({"role_id": X_train["role_id"],"salary": y_train}).groupby("role_id")["salary"].mean())
baseline_predictions = (X_test["role_id"].map(role_mean_salary).fillna(y_train.mean()))

In [63]:
metrics = []
metrics.append({
    "model": "Role Average Baseline",
    "MAE": mean_absolute_error(y_test, baseline_predictions),
    "R2": r2_score(y_test, baseline_predictions)
})

In [64]:
task15_predictions = pd.DataFrame(index=X_test.index)
task15_predictions["actual_salary"] = y_test
task15_predictions["Role Average Baseline"] = baseline_predictions
for name, model in task15_models.items():
    model.fit(X_train, y_train)

    predictions = pd.Series(model.predict(X_test),index=X_test.index)

    task15_predictions[name] = predictions

    metrics.append({
        "model": name,
        "MAE": mean_absolute_error(y_test, predictions),
        "R2": r2_score(y_test, predictions)
    })


In [65]:
metrics_df = pd.DataFrame(metrics)

print("TASK 15 - MODEL PERFORMANCE")
display(metrics_df.round(2))

print("\nPrediction records:", len(task15_predictions))
print("Models trained:", list(task15_models.keys()))

TASK 15 - MODEL PERFORMANCE


,model,MAE,R2
0,Role Average Baseline,115094.24,0.93
1,Linear Regression,81829.72,0.97
2,Gradient Boosting,108464.50,0.94



Prediction records: 587
Models trained: ['Linear Regression', 'Gradient Boosting']


In [66]:
#linear regression performed well 

In [67]:
#fairness audit

In [68]:
fairness_df = salary_audit_df.loc[task15_predictions.index].copy()

In [69]:
fairness_df["actual_salary"] = task15_predictions["actual_salary"]
fairness_df["predicted_salary"] = task15_predictions["Linear Regression"]

In [70]:
def group_gap(data, group_col, actual_col, predicted_col):
    audit = data.copy()
    audit["absolute_error"] = (
        audit[predicted_col] - audit[actual_col]
    ).abs()
    audit["prediction_error"] = (
        audit[predicted_col] - audit[actual_col]
    )
    group_results = (
        audit.groupby(group_col)
        .agg(
            count=(actual_col, "size"),
            mean_actual=(actual_col, "mean"),
            mean_predicted=(predicted_col, "mean"),
            MAE=("absolute_error", "mean"),
            mean_error=("prediction_error", "mean")
        )
        .reset_index()
    )
    gap_summary = pd.DataFrame({
        "metric": ["MAE gap", "Mean error gap"],
        "gap_value": [
            group_results["MAE"].max() - group_results["MAE"].min(),
            group_results["mean_error"].max() - group_results["mean_error"].min()
        ]
    })
    return group_results, gap_summary

In [71]:
gender_results, gender_gaps = group_gap(
    fairness_df,
    "gender",
    "actual_salary",
    "predicted_salary"
)
print("GENDER FAIRNESS RESULTS")
display(gender_results.round(2))
print("GENDER GAP SUMMARY")
display(gender_gaps.round(2))

GENDER FAIRNESS RESULTS


,gender,count,mean_actual,mean_predicted,MAE,mean_error
0,Female,242,847413.37,849348.83,91522.02,1935.46
1,Male,337,839156.23,840833.32,75433.82,1677.09
2,Other,8,696228.00,692435.15,58064.69,-3792.85


GENDER GAP SUMMARY


,metric,gap_value
0,MAE gap,33457.33
1,Mean error gap,5728.31


In [72]:
career_results, career_gaps = group_gap(
    fairness_df,
    "career_gap_group",
    "actual_salary",
    "predicted_salary"
)
print("\nCAREER GAP FAIRNESS RESULTS")
display(career_results.round(2))
print("CAREER GAP SUMMARY")
display(career_gaps.round(2))


CAREER GAP FAIRNESS RESULTS


,career_gap_group,count,mean_actual,mean_predicted,MAE,mean_error
0,Career gap,79,814216.10,844292.47,87864.46,30076.37
1,No career gap,508,844717.39,842015.00,80891.24,-2702.39


CAREER GAP SUMMARY


,metric,gap_value
0,MAE gap,6973.22
1,Mean error gap,32778.76


In [73]:
#generate recommended pay bands

In [79]:
recommendations = pd.DataFrame(index=task15_predictions.index)
recommendations["hire_id"] = salary_audit_df.loc[
    recommendations.index, "hire_id"]
recommendations["role_id"] = salary_audit_df.loc[
    recommendations.index, "role_id"]
recommendations["job_level"] = salary_audit_df.loc[
    recommendations.index, "job_level"]
recommendations["predicted_salary"] = (
    task15_predictions["Linear Regression"])


Recommendation records: 587


In [76]:
recommendations["low_band"] = (
    recommendations["predicted_salary"] * 0.90
).round(0)

recommendations["mid_band"] = (
    recommendations["predicted_salary"]
).round(0)

recommendations["high_band"] = (
    recommendations["predicted_salary"] * 1.10
).round(0)

In [77]:
salary_columns = [
    "predicted_salary",
    "low_band",
    "mid_band",
    "high_band"
]

In [78]:
recommendations[salary_columns] = (
    recommendations[salary_columns].round(0).astype("int64")
)
print("RECOMMENDED PAY BAND SAMPLE")
display(recommendations.head(10))
print("\nRECOMMENDATION SUMMARY")
display(
    recommendations[
        ["low_band", "mid_band", "high_band"]
    ].describe().round(2)
)

RECOMMENDED PAY BAND SAMPLE


,hire_id,role_id,job_level,predicted_salary,low_band,mid_band,high_band
2418,HIRE-1685,R033,1,65346,58811,65346,71881
2608,HIRE-1875,R005,3,1405865,1265278,1405865,1546452
2922,HIRE-2189,R004,2,448255,403430,448255,493081
685,EMP-0697,R008,5,3410985,3069886,3410985,3752084
1041,HIRE-0308,R040,2,580505,522454,580505,638556
1637,HIRE-0904,R003,2,848680,763812,848680,933548
2228,HIRE-1495,R003,2,618980,557082,618980,680878
410,EMP-0422,R051,2,401832,361649,401832,442015
2916,HIRE-2183,R006,3,1210505,1089454,1210505,1331556
436,EMP-0448,R007,4,2064789,1858310,2064789,2271268



RECOMMENDATION SUMMARY


,low_band,mid_band,high_band
count,587.00,587.00,587.00
mean,758089.37,842321.52,926553.71
std,588945.44,654383.85,719822.26
min,-9643.00,-10714.00,-11785.00
25%,347193.00,385770.00,424347.00
50%,573603.00,637337.00,701071.00
75%,1050662.00,1167402.50,1284143.00
max,3069886.00,3410985.00,3752084.00


In [80]:
len(recommendations)

587

In [81]:
# negative value diagonose

In [82]:
negative_predictions = recommendations[recommendations["predicted_salary"] < 0].copy()

In [83]:
print("Negative prediction count:", len(negative_predictions))
print("Percentage of test predictions:",round(len(negative_predictions) / len(recommendations) * 100, 2), "%")
print("\nNEGATIVE PREDICTION DETAILS")
display(negative_predictions)
print("\nLOWEST PREDICTIONS")
display(recommendations.nsmallest(10, "predicted_salary"))

Negative prediction count: 1
Percentage of test predictions: 0.17 %

NEGATIVE PREDICTION DETAILS


,hire_id,role_id,job_level,predicted_salary
12,EMP-0014,R034,2,-10714.123207



LOWEST PREDICTIONS


,hire_id,role_id,job_level,predicted_salary
12,EMP-0014,R034,2,-10714.123207
56,EMP-0062,R050,1,3433.812908
2770,HIRE-2037,R033,1,29165.949244
1488,HIRE-0755,R033,1,38069.727735
135,EMP-0146,R033,1,62954.534528
2418,HIRE-1685,R033,1,65345.678625
120,EMP-0131,R033,1,71673.112010
1091,HIRE-0358,R033,1,98505.077235
2685,HIRE-1952,R044,1,100018.975704
29,EMP-0031,R051,2,100829.540589


In [84]:
guardrails = salary_audit_df.loc[
    recommendations.index,
    ["band_min_2026", "band_max_2026"]]
recommendations["raw_prediction"] = (
    task15_predictions["Linear Regression"]
)
below_min = (
    recommendations["raw_prediction"] < guardrails["band_min_2026"]
)
above_max = (
    recommendations["raw_prediction"] > guardrails["band_max_2026"]
)
print("Predictions below role minimum:", int(below_min.sum()))
print("Predictions above role maximum:", int(above_max.sum()))
# Constrain the midpoint prediction to the role-specific range
recommendations["mid_band"] = np.clip(
    recommendations["raw_prediction"],
    guardrails["band_min_2026"],
    guardrails["band_max_2026"]
).round(0)
# Build low/mid/high ranges, also respecting the guardrails
recommendations["low_band"] = np.maximum(
    recommendations["mid_band"] * 0.90,
    guardrails["band_min_2026"]
).round(0)
recommendations["high_band"] = np.minimum(
    recommendations["mid_band"] * 1.10,
    guardrails["band_max_2026"]
).round(0)
recommendations["predicted_salary"] = recommendations["mid_band"]
salary_columns = [
    "predicted_salary",
    "low_band",
    "mid_band",
    "high_band"
]

recommendations[salary_columns] = (
    recommendations[salary_columns].round(0).astype("int64")
)

Predictions below role minimum: 256
Predictions above role maximum: 9

CORRECTED PAY-BAND SAMPLE


,hire_id,role_id,job_level,predicted_salary,raw_prediction,mid_band,low_band,high_band
2418,HIRE-1685,R033,1,208000,6.534568e+04,208000,208000,228800
2608,HIRE-1875,R005,3,1405865,1.405865e+06,1405865,1280000,1546452
2922,HIRE-2189,R004,2,600000,4.482550e+05,600000,600000,660000
685,EMP-0697,R008,5,3410985,3.410985e+06,3410985,3069886,3752084
1041,HIRE-0308,R040,2,580505,5.805050e+05,580505,522454,638556
1637,HIRE-0904,R003,2,848680,8.486800e+05,848680,763812,933548
2228,HIRE-1495,R003,2,720000,6.189802e+05,720000,720000,792000
410,EMP-0422,R051,2,401832,4.018316e+05,401832,361649,442015
2916,HIRE-2183,R006,3,1210505,1.210505e+06,1210505,1200000,1331556
436,EMP-0448,R007,4,2064789,2.064789e+06,2064789,2000000,2271268



CORRECTED PAY-BAND SUMMARY


,predicted_salary,low_band,mid_band,high_band
count,587.00,587.00,587.00,587.00
mean,884627.24,845111.11,884627.24,971860.41
std,641793.35,615517.16,641793.35,706804.34
min,208000.00,208000.00,208000.00,228800.00
25%,408304.50,400000.00,408304.50,442734.00
50%,680000.00,680000.00,680000.00,748000.00
75%,1201845.00,1200000.00,1201845.00,1322029.50
max,3410985.00,3069886.00,3410985.00,3752084.00



Validation checks:
Negative mid-band values: 0
Low band below role minimum: 0
High band above role maximum: 0


In [85]:
print("\nCORRECTED PAY-BAND SAMPLE")
display(recommendations.head(10))
print("\nCORRECTED PAY-BAND SUMMARY")
display(recommendations[salary_columns].describe().round(2))
print("\nValidation checks:")
print("Negative mid-band values:", int((recommendations["mid_band"] < 0).sum()))
print(
    "Low band below role minimum:",
    int((recommendations["low_band"] < guardrails["band_min_2026"]).sum())
)
print(
    "High band above role maximum:",
    int((recommendations["high_band"] > guardrails["band_max_2026"]).sum())
)


CORRECTED PAY-BAND SAMPLE


,hire_id,role_id,job_level,predicted_salary,raw_prediction,mid_band,low_band,high_band
2418,HIRE-1685,R033,1,208000,6.534568e+04,208000,208000,228800
2608,HIRE-1875,R005,3,1405865,1.405865e+06,1405865,1280000,1546452
2922,HIRE-2189,R004,2,600000,4.482550e+05,600000,600000,660000
685,EMP-0697,R008,5,3410985,3.410985e+06,3410985,3069886,3752084
1041,HIRE-0308,R040,2,580505,5.805050e+05,580505,522454,638556
1637,HIRE-0904,R003,2,848680,8.486800e+05,848680,763812,933548
2228,HIRE-1495,R003,2,720000,6.189802e+05,720000,720000,792000
410,EMP-0422,R051,2,401832,4.018316e+05,401832,361649,442015
2916,HIRE-2183,R006,3,1210505,1.210505e+06,1210505,1200000,1331556
436,EMP-0448,R007,4,2064789,2.064789e+06,2064789,2000000,2271268



CORRECTED PAY-BAND SUMMARY


,predicted_salary,low_band,mid_band,high_band
count,587.00,587.00,587.00,587.00
mean,884627.24,845111.11,884627.24,971860.41
std,641793.35,615517.16,641793.35,706804.34
min,208000.00,208000.00,208000.00,228800.00
25%,408304.50,400000.00,408304.50,442734.00
50%,680000.00,680000.00,680000.00,748000.00
75%,1201845.00,1200000.00,1201845.00,1322029.50
max,3410985.00,3069886.00,3410985.00,3752084.00



Validation checks:
Negative mid-band values: 0
Low band below role minimum: 0
High band above role maximum: 0


In [86]:
# Generate recommdation for all records

In [87]:
from sklearn.base import clone


In [88]:
final_pay_model = clone(task15_models["Linear Regression"])
final_pay_model.fit(X, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](8,)","['department_id','role_id','job_level',...,'prior_experience_years', 'skills_count','market_index']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,8
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numerical', ...)]"
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0
,"remainder remain

In [89]:
all_predictions = final_pay_model.predict(X)

In [90]:
final_recommendations = salary_audit_df[
    ["hire_id", "role_id", "job_level","band_min_2026", "band_max_2026"]].copy()

In [91]:
final_recommendations["raw_prediction"] = all_predictions


In [92]:
final_recommendations["mid_band"] = np.clip(
    final_recommendations["raw_prediction"],
    final_recommendations["band_min_2026"],
    final_recommendations["band_max_2026"]
).round(0)

final_recommendations["low_band"] = np.maximum(
    final_recommendations["mid_band"] * 0.90,
    final_recommendations["band_min_2026"]
).round(0)

final_recommendations["high_band"] = np.minimum(
    final_recommendations["mid_band"] * 1.10,
    final_recommendations["band_max_2026"]
).round(0)

final_recommendations["guardrail_adjusted"] = (
    (final_recommendations["raw_prediction"] <
     final_recommendations["band_min_2026"]) |
    (final_recommendations["raw_prediction"] >
     final_recommendations["band_max_2026"])
)

In [93]:
final_recommendations = final_recommendations[
    [
        "hire_id",
        "role_id",
        "job_level",
        "raw_prediction",
        "low_band",
        "mid_band",
        "high_band",
        "guardrail_adjusted"
    ]
]

In [94]:
display(final_recommendations.head(10))

,hire_id,role_id,job_level,raw_prediction,low_band,mid_band,high_band,guardrail_adjusted
0,EMP-0001,R002,1,5.143343e+04,400000.0,400000.0,440000.0,True
1,EMP-0002,R007,4,1.708575e+06,2000000.0,2000000.0,2200000.0,True
2,EMP-0003,R051,2,1.950193e+04,360000.0,360000.0,396000.0,True
3,EMP-0004,R017,3,8.568047e+05,1200000.0,1200000.0,1320000.0,True
4,EMP-0005,R033,1,-1.397772e+05,208000.0,208000.0,228800.0,True
5,EMP-0007,R006,3,8.841818e+05,1200000.0,1200000.0,1320000.0,True
6,EMP-0008,R034,2,-4.184752e+04,304000.0,304000.0,334400.0,True
7,EMP-0009,R025,4,1.472623e+06,1760000.0,1760000.0,1936000.0,True
8,EMP-0010,R005,3,9.968173e+05,1280000.0,1280000.0,1408000.0,True
9,EMP-0011,R039,1,-1.543650e+04,272000.0,272000.0,299200.0,True


In [95]:
display(
    final_recommendations[
        ["low_band", "mid_band", "high_band"]
    ].describe().round(2)
)


,low_band,mid_band,high_band
count,2934.00,2934.00,2934.00
mean,841356.05,879025.32,965545.18
std,597700.96,621038.62,684130.26
min,208000.00,208000.00,228800.00
25%,400000.00,416000.00,456622.50
50%,680000.00,680000.00,748000.00
75%,1200000.00,1201346.25,1321480.50
max,3050793.00,3389770.00,3728747.00


In [96]:
final_recommendations["guardrail_adjusted"].sum()

np.int64(1425)

In [97]:
(final_recommendations["mid_band"] <= 0).sum()

np.int64(0)

In [99]:
#save the task 15 evidence 

In [102]:
from pathlib import Path
reports_dir = Path("/workspace/reports")
reports_dir.mkdir(parents=True, exist_ok=True)
final_recommendations.to_csv(
    reports_dir / "task15_pay_band_recommendations.csv",
    index=False
)
metrics_df.to_csv(
    reports_dir / "task15_model_metrics.csv",
    index=False
)
gender_results.to_csv(
    reports_dir / "task15_gender_fairness.csv",
    index=False
)
gender_gaps.to_csv(
    reports_dir / "task15_gender_gaps.csv",
    index=False
)
career_results.to_csv(
    reports_dir / "task15_career_gap_fairness.csv",
    index=False
)
career_gaps.to_csv(
    reports_dir / "task15_career_gap_gaps.csv",
    index=False
)
guardrail_summary = pd.DataFrame({
    "metric": [
        "Total recommendations",
        "Guardrail-adjusted recommendations",
        "Unadjusted recommendations",
        "Adjustment percentage",
        "Invalid final recommendations"
    ],
    "value": [
        len(final_recommendations),
        int(final_recommendations["guardrail_adjusted"].sum()),
        int((~final_recommendations["guardrail_adjusted"]).sum()),
        round(final_recommendations["guardrail_adjusted"].mean() * 100, 2),
        int((final_recommendations["mid_band"] <= 0).sum())
    ]
})
guardrail_summary.to_csv(
    reports_dir / "task15_guardrail_summary.csv",
    index=False
)
for file in sorted(reports_dir.glob("task15_*.csv")):
    print(file.name, "|", file.stat().st_size, "bytes")

TASK 15 EVIDENCE FILES SAVED
task15_career_gap_fairness.csv | 243 bytes
task15_career_gap_gaps.csv | 77 bytes
task15_gender_fairness.csv | 293 bytes
task15_gender_gaps.csv | 77 bytes
task15_guardrail_summary.csv | 180 bytes
task15_model_metrics.csv | 182 bytes
task15_pay_band_recommendations.csv | 201071 bytes

GUARDRAIL SUMMARY


In [101]:
display(guardrail_summary)

,metric,value
0,Total recommendations,2934.00
1,Guardrail-adjusted recommendations,1425.00
2,Unadjusted recommendations,1509.00
3,Adjustment percentage,48.57
4,Invalid final recommendations,0.00


In [103]:
# meaning of gap 

In [ ]:
'''
-A group gap measures the difference in model performance or prediction errors between different groups.
-In task we used (MAE) mean squared error and Mean Error to evaluate whether the salary prediction model performs 
    differently across gender groups and employees with or without career gaps.
        >MAE gap: Difference between the highest and lowest group MAE. A larger gap indicates that the model's prediction errors vary more across groups.
        >Mean error gap: Difference between the highest and lowest group mean prediction errors. It helps identify whether the model systematically overestimates or underestimates salaries for particular groups.
we evaluated salary prediciton accuracy,measured group level fairness gaps , exculed protected sttributes from model training and intriduced salary ranges with role specific guardials 
'''
        